# Preparación 

El siguiente cuaderno presenta técnicas para transformar un conjunto de datos. Al igual que los cuadernos 6 y 7, se trabajará con el dataset  **NSL-KDD**, una versión mejorada de KDD’99 diseñada para evaluar sistemas de detección de intrusos (IDS) en tráfico de red.  

De forma general, NSL-KDD contiene registros de conexiones etiquetadas como **tráfico normal** o **tipos de ataque**, por lo que es útil como conjunto de referencia para tareas de clasificación, análisis exploratorio y comparación de modelos.

Aunque no representa por completo las redes reales actuales, NSL-KDD sigue siendo ampliamente utilizado en investigación por su estructura, tamaño manejable y etiquetas estandarizadas.

### Fuente del dataset
El siguiente dataset puede descargarse en este enlace: **https://www.kaggle.com/datasets/hassan06/nslkdd**. 

## 1. Import de librerías

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler

### Funciones auxiliares

In [2]:
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

## 2. Lectura de datos

In [3]:
df = pd.read_csv("data/KDDTrain+.csv")

In [4]:
df

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,class,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125968,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.06,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20
125969,8,udp,private,SF,105,145,0,0,0,0,...,0.96,0.01,0.01,0.00,0.00,0.00,0.00,0.00,normal,21
125970,0,tcp,smtp,SF,2231,384,0,0,0,0,...,0.12,0.06,0.00,0.00,0.72,0.00,0.01,0.00,normal,18
125971,0,tcp,klogin,S0,0,0,0,0,0,0,...,0.03,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,20


## 2. División del conjunto de datos

In [5]:
train_set, val_set, test_set = train_val_test_split(df, stratify='protocol_type')

In [6]:
train_set.shape[0], val_set.shape[0], test_set.shape[0]

(75583, 25195, 25195)

## 3. Limpieza de datos

In [7]:
# Separar características (X) y etiquetas (y) para el conjunto de entrenamiento
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

In [8]:
# Reemplazar valores atípicos en 'src_bytes' y 'dst_bytes' con valores nulos
X_train.loc[(X_train["src_bytes"]>400) & (X_train["src_bytes"]<800), "src_bytes"] = np.nan
X_train.loc[(X_train["dst_bytes"]>500) & (X_train["dst_bytes"]<2000), "dst_bytes"] = np.nan
X_train

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,NaN,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64559,0,tcp,systat,S0,0.0,0.0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.00,0.0,0.0,20
67272,0,tcp,http,SF,210.0,NaN,0,0,0,0,...,255,1.00,0.00,0.01,0.02,0.02,0.01,0.0,0.0,21
32452,3,tcp,smtp,SF,889.0,328.0,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.00,0.0,0.0,21
112657,0,tcp,http,SF,284.0,444.0,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21


In [9]:
# Comprobar si hay valores nulos en el conjunto de entrenamiento
X_train.isna().any()

duration                       False
protocol_type                  False
service                        False
flag                           False
src_bytes                       True
dst_bytes                       True
land                           False
wrong_fragment                 False
urgent                         False
hot                            False
num_failed_logins              False
logged_in                      False
num_compromised                False
root_shell                     False
su_attempted                   False
num_root                       False
num_file_creations             False
num_shells                     False
num_access_files               False
num_outbound_cmds              False
is_host_login                  False
is_guest_login                 False
count                          False
srv_count                      False
serror_rate                    False
srv_serror_rate                False
rerror_rate                    False
s

In [10]:
# Mostrar filas con valores nulos
null_data  = X_train[X_train.isnull().any(axis=1)]
null_data

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,NaN,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
108116,0,tcp,http,SF,304.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
64957,1,tcp,smtp,SF,NaN,329.0,0,0,0,0,...,181,0.65,0.03,0.01,0.01,0.02,0.02,0.0,0.0,21
100052,0,tcp,http,SF,206.0,NaN,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21
99158,0,tcp,http,SF,291.0,NaN,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117260,0,tcp,http,SF,321.0,NaN,0,0,0,0,...,255,1.00,0.00,0.50,0.02,0.00,0.00,0.0,0.0,21
110723,0,tcp,http,SF,361.0,NaN,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
58053,0,tcp,http,SF,202.0,NaN,0,0,0,0,...,255,1.00,0.00,0.01,0.01,0.00,0.00,0.0,0.0,21
70184,0,tcp,http,SF,315.0,NaN,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21


Existen tres opciones para resolver este problema:

* Eliminar las filas correspondientes
* Eliminar el atributo (columna) correspondiente
* Rellenarlos con un valor determinado (zero, media ...)

### Opción 1: Eliminar filas correspondientes

In [11]:
# Crear una copia de X_train para manipular los datos sin afectar al original
X_train_copy = X_train.copy()

In [12]:
# Eliminar filas con valores nulos en 'src_bytes' y 'dst_bytes'
X_train_copy.dropna(subset=["src_bytes", "dst_bytes"], inplace=True)
X_train_copy

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.0,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.0,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.0,0.0,0.0,16
98007,0,udp,domain_u,SF,46.0,139.0,0,0,0,0,...,254,1.00,0.01,0.00,0.00,0.00,0.0,0.0,0.0,20
16447,0,tcp,smtp,SF,1790.0,363.0,0,0,0,0,...,137,0.55,0.04,0.01,0.01,0.00,0.0,0.0,0.0,21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90665,0,tcp,ftp_data,S0,0.0,0.0,0,0,0,0,...,63,0.25,0.02,0.02,0.00,1.00,1.0,0.0,0.0,18
64559,0,tcp,systat,S0,0.0,0.0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.0,0.0,0.0,20
32452,3,tcp,smtp,SF,889.0,328.0,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.0,0.0,0.0,21
112657,0,tcp,http,SF,284.0,444.0,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.0,0.0,0.0,21


In [13]:
print(f"Se eliminaron {len(X_train) - len(X_train_copy)} filas")

Se eliminaron 9886 filas


### Opción 2: Eliminar atributos con valores nulos

In [14]:
X_train_copy = X_train.copy()

In [15]:
X_train_copy.drop(["src_bytes", "dst_bytes"], axis=1, inplace=True)
X_train_copy

,duration,protocol_type,service,flag,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,0,0,0,0,0,1,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0,0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,0,0,0,0,0,1,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0,0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,0,0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64559,0,tcp,systat,S0,0,0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.00,0.0,0.0,20
67272,0,tcp,http,SF,0,0,0,0,0,1,...,255,1.00,0.00,0.01,0.02,0.02,0.01,0.0,0.0,21
32452,3,tcp,smtp,SF,0,0,0,0,0,1,...,155,0.64,0.04,0.01,0.01,0.01,0.00,0.0,0.0,21
112657,0,tcp,http,SF,0,0,0,0,0,1,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21


In [16]:
print(f"Se eliminaron {len(list(X_train)) - len(list(X_train_copy))} atributos")

Se eliminaron 2 atributos


### Opción 3: Rellenar con un valor determinado

In [17]:
X_train_copy = X_train.copy()

In [18]:
# Se rellenarán los valores nulos con la media de los valores del atributo
mean_srcbytes = X_train_copy["src_bytes"].mean()
mean_dstbytes = X_train_copy["dst_bytes"].mean()

X_train_copy["src_bytes"] = X_train_copy["src_bytes"].fillna(mean_srcbytes)
X_train_copy["dst_bytes"] = X_train_copy["dst_bytes"].fillna(mean_dstbytes)

X_train_copy

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,66914.530762,53508.000000,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0.000000,0.000000,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304.000000,9181.334754,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0.000000,0.000000,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.000000,0.000000,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64559,0,tcp,systat,S0,0.000000,0.000000,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.00,0.0,0.0,20
67272,0,tcp,http,SF,210.000000,9181.334754,0,0,0,0,...,255,1.00,0.00,0.01,0.02,0.02,0.01,0.0,0.0,21
32452,3,tcp,smtp,SF,889.000000,328.000000,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.00,0.0,0.0,21
112657,0,tcp,http,SF,284.000000,444.000000,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21


In [19]:
X_train_copy = X_train.copy()

In [20]:
# Ahora se probará con la mediana en lugar de la media para rellenar los valores nulos, debido 
# a que la media puede ser afectada por los valores atípicos, mientras que la mediana no lo es.

median_srcbytes = X_train_copy["src_bytes"].median()
median_dstbytes = X_train_copy["dst_bytes"].median()

X_train_copy["src_bytes"] = X_train_copy["src_bytes"].fillna(median_srcbytes)
X_train_copy["dst_bytes"] = X_train_copy["dst_bytes"].fillna(median_dstbytes)

X_train_copy

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,43.0,53508.0,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304.0,0.0,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0.0,0.0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8.0,0.0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64559,0,tcp,systat,S0,0.0,0.0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.00,0.0,0.0,20
67272,0,tcp,http,SF,210.0,0.0,0,0,0,0,...,255,1.00,0.00,0.01,0.02,0.02,0.01,0.0,0.0,21
32452,3,tcp,smtp,SF,889.0,328.0,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.00,0.0,0.0,21
112657,0,tcp,http,SF,284.0,444.0,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21


In [21]:
# Existe otra alternativa: usar SimpleImputer de sklearn para rellenar 
# los valores nulos con la media o la mediana de cada atributo.

imputer = SimpleImputer(strategy="median")

In [22]:
# La clase imputer no admite valores categoricos, por lo que se eliminan los atributos categoricos
X_train_copy_num = X_train_copy.select_dtypes(exclude=['object'])
X_train_copy_num.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75583 entries, 113467 to 99030
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   duration                     75583 non-null  int64  
 1   src_bytes                    75583 non-null  float64
 2   dst_bytes                    75583 non-null  float64
 3   land                         75583 non-null  int64  
 4   wrong_fragment               75583 non-null  int64  
 5   urgent                       75583 non-null  int64  
 6   hot                          75583 non-null  int64  
 7   num_failed_logins            75583 non-null  int64  
 8   logged_in                    75583 non-null  int64  
 9   num_compromised              75583 non-null  int64  
 10  root_shell                   75583 non-null  int64  
 11  su_attempted                 75583 non-null  int64  
 12  num_root                     75583 non-null  int64  
 13  num_file_creatio

In [23]:
# Se le proporcionan los atributos numericos para que calcule los valores
imputer.fit(X_train_copy_num)

,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [24]:
# Rellenar los valores nulos con la mediana calculada por el imputer
X_train_copy_num_nonan = imputer.transform(X_train_copy_num)


In [25]:
# Se transforma el resultado a un dataframe
X_train_copy = pd.DataFrame(X_train_copy_num_nonan, columns=X_train_copy_num.columns)

X_train_copy

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
0,0.0,43.0,53508.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,255.0,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21.0
2,0.0,304.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,255.0,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,15.0,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21.0
4,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.0,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75578,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,20.0,0.08,0.06,0.00,0.00,1.00,1.00,0.0,0.0,20.0
75579,0.0,210.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,255.0,1.00,0.00,0.01,0.02,0.02,0.01,0.0,0.0,21.0
75580,3.0,889.0,328.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,155.0,0.64,0.04,0.01,0.01,0.01,0.00,0.0,0.0,21.0
75581,0.0,284.0,444.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,255.0,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21.0


## 4. Transformación de atributos categóricos a numéricos

In [26]:
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

In [27]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75583 entries, 113467 to 99030
Data columns (total 42 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   duration                     75583 non-null  int64  
 1   protocol_type                75583 non-null  object 
 2   service                      75583 non-null  object 
 3   flag                         75583 non-null  object 
 4   src_bytes                    75583 non-null  int64  
 5   dst_bytes                    75583 non-null  int64  
 6   land                         75583 non-null  int64  
 7   wrong_fragment               75583 non-null  int64  
 8   urgent                       75583 non-null  int64  
 9   hot                          75583 non-null  int64  
 10  num_failed_logins            75583 non-null  int64  
 11  logged_in                    75583 non-null  int64  
 12  num_compromised              75583 non-null  int64  
 13  root_shell      

In [28]:
# Convertir valores categóricos a numéricos usando el método factorize de pandas
protocol_type = X_train['protocol_type']
protocol_type_encoded, categorias = protocol_type.factorize()

In [29]:
for i in range(10):
    print(protocol_type.iloc[i], "=", protocol_type_encoded[i])

tcp = 0
tcp = 0
tcp = 0
tcp = 0
icmp = 1
udp = 2
tcp = 0
tcp = 0
tcp = 0
tcp = 0


In [30]:
print(categorias)

Index(['tcp', 'icmp', 'udp'], dtype='object')


In [31]:
# Otra alternativa es usar OrdinalEncoder de sklearn para convertir los valores categoricos a numericos
protocol_type = X_train[['protocol_type']]
ordinal_encoder = OrdinalEncoder()

In [32]:
# Utiliza el mismo método que factorize
for i in range(10):
    print(protocol_type["protocol_type"].iloc[i], "=", protocol_type_encoded[i])


tcp = 0
tcp = 0
tcp = 0
tcp = 0
icmp = 1
udp = 2
tcp = 0
tcp = 0
tcp = 0
tcp = 0


En este caso, no tendría mucho sentido usar OrdinalEncoder, ya que ciertos algoritmos de ML que funcionan mediendo la similitud de dos puntos por  distancia, este caso (para estos valores categóricos), no tiene sentido.

In [33]:
# Se genera una matriz binaria para cada valor de 'protocol_type' usando OneHotEncoder de sklearn
# La sparse matrix solo almacena la posicion de los valores que no son '0' para ahorrar memoria

protocol_type = X_train[['protocol_type']]

oh_encoder = OneHotEncoder()
protocol_type_oh = oh_encoder.fit_transform(protocol_type)
protocol_type_oh

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 75583 stored elements and shape (75583, 3)>

In [34]:
protocol_type_oh.toarray()

array([[0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.],
       ...,
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 1., 0.]], shape=(75583, 3))

In [35]:
for i in range(10):
    print(protocol_type["protocol_type"].iloc[i], "=", protocol_type_oh.toarray()[i])

tcp = [0. 1. 0.]
tcp = [0. 1. 0.]
tcp = [0. 1. 0.]
tcp = [0. 1. 0.]
icmp = [1. 0. 0.]
udp = [0. 0. 1.]
tcp = [0. 1. 0.]
tcp = [0. 1. 0.]
tcp = [0. 1. 0.]
tcp = [0. 1. 0.]


In [36]:
print(oh_encoder.categories_)

[array(['icmp', 'tcp', 'udp'], dtype=object)]


Al particionar el conjunto de datos o realizar una predicción con nuevos ejemplos, aparecen nuevos valores para determinadas categorías que producirán un error en la función **transform()**. OneHotEncoding proporciona el parámetro **handle_uknown** ya sea para generar un error o ignorar si una característica categórica desconocida está presente durante la transformación (el valor predeterminado es lanzar un error). 

Cuando este parámetro se establece en "ignorar" y se encuentra una categoría desconocida durante la transformación, las columnas codificadas resultantes para esta característica serán todo ceros. En la transformación inversa, una categoría desconocida se denotará como None.

In [37]:
oh_encoder = OneHotEncoder(handle_unknown='ignore')

In [38]:
# Uso de Get Dummies de pandas para convertir los valores categoricos a numericos
# Aplica OneHotEncoding a varias columnas a la vez y maneja automáticamente los valores desconocidos en el conjunto de prueba

pd.get_dummies(X_train['protocol_type'])

,icmp,tcp,udp
113467,False,True,False
31899,False,True,False
108116,False,True,False
89913,False,True,False
106319,True,False,False
...,...,...,...
64559,False,True,False
67272,False,True,False
32452,False,True,False
112657,False,True,False


## 5. Escalado del conjunto de datos

In [39]:
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

En Machine Learning, conviene escalar las características cuando tienen rangos muy distintos para mejorar el rendimiento del modelo. Para eso, se aplica:  
* Normalización: transforma los valores al rango [0, 1].  
* Estandarización: centra y escala los valores (normalmente media 0 y desviación típica 1), sin rango fijo.  

El escalado se aplica solo a las variables de entrada, nunca a las etiquetas.  
Se ajusta con entrenamiento y luego se aplica a prueba con esos mismos parámetros.

In [40]:
# Se escalan los atributos 'src_bytes' y 'dst_bytes' usando RobustScaler de sklearn,
# que es menos sensible a los valores atípicos
scale_attrs = X_train[['src_bytes', 'dst_bytes']]

robust_scaler = RobustScaler()
X_train_scaled = robust_scaler.fit_transform(scale_attrs)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=['src_bytes', 'dst_bytes'])

In [41]:
# Se reemplazan los atributos originales por los escalados
X_train_scaled

,src_bytes,dst_bytes
0,1.324818,101.920000
1,-0.160584,0.000000
2,0.948905,1.211429
3,-0.160584,0.000000
4,-0.131387,0.000000
...,...,...
75578,-0.160584,0.000000
75579,0.605839,1.401905
75580,3.083942,0.624762
75581,0.875912,0.845714


In [42]:
X_train

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty
113467,0,tcp,http,SF,407,53508,0,0,0,0,...,255,1.00,0.00,0.11,0.03,0.00,0.00,0.0,0.0,21
31899,0,tcp,private,S0,0,0,0,0,0,0,...,4,0.02,0.05,0.00,0.00,1.00,1.00,0.0,0.0,21
108116,0,tcp,http,SF,304,636,0,0,0,0,...,255,1.00,0.00,0.03,0.06,0.00,0.00,0.0,0.0,21
89913,0,tcp,private,S0,0,0,0,0,0,0,...,15,0.06,0.07,0.00,0.00,1.00,1.00,0.0,0.0,21
106319,0,icmp,eco_i,SF,8,0,0,0,0,0,...,7,1.00,0.00,1.00,0.57,0.00,0.00,0.0,0.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64559,0,tcp,systat,S0,0,0,0,0,0,0,...,20,0.08,0.06,0.00,0.00,1.00,1.00,0.0,0.0,20
67272,0,tcp,http,SF,210,736,0,0,0,0,...,255,1.00,0.00,0.01,0.02,0.02,0.01,0.0,0.0,21
32452,3,tcp,smtp,SF,889,328,0,0,0,0,...,155,0.64,0.04,0.01,0.01,0.01,0.00,0.0,0.0,21
112657,0,tcp,http,SF,284,444,0,0,0,0,...,255,1.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,21
